In [59]:
import yfinance as yf
import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt
from scipy.stats import entropy
from datetime import datetime

In [60]:
# fetch data
ticker_symbol = "AAPL"
start_date = "2016-10-01"
end_date = "2024-10-01"
df = yf.download(ticker_symbol, start=start_date, end=end_date).reset_index()

[*********************100%***********************]  1 of 1 completed


# Days to next FOMC meeting

In [61]:
# List of FOMC meeting dates from 2016 to 2024 (upcoming and past)
fomc_meeting_dates = [
    '2016-01-27', '2016-03-16', '2016-04-27', '2016-06-15', '2016-07-27', '2016-09-21', '2016-11-02', '2016-12-14',
    '2017-02-01', '2017-03-15', '2017-05-03', '2017-06-14', '2017-07-26', '2017-09-20', '2017-11-01', '2017-12-13',
    '2018-01-31', '2018-03-21', '2018-05-02', '2018-06-13', '2018-08-01', '2018-09-26', '2018-11-08', '2018-12-19',
    '2019-01-30', '2019-03-20', '2019-05-01', '2019-06-19', '2019-07-31', '2019-09-18', '2019-10-30', '2019-12-11',
    '2020-01-29', '2020-03-18', '2020-04-29', '2020-06-10', '2020-07-29', '2020-09-16', '2020-11-05', '2020-12-16',
    '2021-01-27', '2021-03-17', '2021-04-28', '2021-06-16', '2021-07-28', '2021-09-22', '2021-11-03', '2021-12-15',
    '2022-01-26', '2022-03-16', '2022-05-04', '2022-06-15', '2022-07-27', '2022-09-21', '2022-11-02', '2022-12-14',
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-14', '2023-07-26', '2023-09-20', '2023-11-01', '2023-12-13',
    '2024-01-31', '2024-03-20', '2024-05-01', '2024-06-19', '2024-07-31', '2024-09-18', '2024-11-06', '2024-12-18'
]

fomc_meeting_dates = pd.to_datetime(fomc_meeting_dates)
df['Date'] = pd.to_datetime(df['Date']).dt.tz_convert(None)
# Calculate the days to the closest FOMC meeting
df['Days to Next FOMC'] = \
    df['Date'].apply(lambda x: (fomc_meeting_dates[fomc_meeting_dates > x].min() - x).days)


# Days to next CPI release

In [62]:
cpi_release_dates = [
    '2016-01-19', '2016-02-19', '2016-03-16', '2016-04-14', '2016-05-17', '2016-06-16', '2016-07-15', '2016-08-16', '2016-09-16', '2016-10-18', '2016-11-17', '2016-12-15',
    '2017-01-18', '2017-02-15', '2017-03-15', '2017-04-14', '2017-05-16', '2017-06-15', '2017-07-14', '2017-08-15', '2017-09-15', '2017-10-13', '2017-11-15', '2017-12-13',
    '2018-01-12', '2018-02-14', '2018-03-13', '2018-04-11', '2018-05-10', '2018-06-12', '2018-07-12', '2018-08-10', '2018-09-13', '2018-10-11', '2018-11-14', '2018-12-12',
    '2019-01-11', '2019-02-13', '2019-03-12', '2019-04-10', '2019-05-10', '2019-06-12', '2019-07-11', '2019-08-13', '2019-09-12', '2019-10-10', '2019-11-13', '2019-12-11',
    '2020-01-14', '2020-02-13', '2020-03-11', '2020-04-10', '2020-05-12', '2020-06-10', '2020-07-14', '2020-08-12', '2020-09-11', '2020-10-13', '2020-11-12', '2020-12-11',
    '2021-01-13', '2021-02-10', '2021-03-10', '2021-04-13', '2021-05-12', '2021-06-10', '2021-07-13', '2021-08-11', '2021-09-14', '2021-10-13', '2021-11-10', '2021-12-10',
    '2022-01-12', '2022-02-10', '2022-03-10', '2022-04-12', '2022-05-11', '2022-06-10', '2022-07-13', '2022-08-10', '2022-09-13', '2022-10-12', '2022-11-10', '2022-12-13',
    '2023-01-12', '2023-02-14', '2023-03-14', '2023-04-12', '2023-05-10', '2023-06-13', '2023-07-12', '2023-08-10', '2023-09-13', '2023-10-12', '2023-11-14', '2023-12-12',
    '2024-01-10', '2024-02-14', '2024-03-13', '2024-04-10', '2024-05-15', '2024-06-12', '2024-07-17', '2024-08-14', '2024-09-11', '2024-10-10', '2024-11-13', '2024-12-11'
]

cpi_release_dates = pd.to_datetime(cpi_release_dates)
# Calculate the days to the closest FOMC meeting
df['Days to Next CPI'] = \
    df['Date'].apply(lambda x: (cpi_release_dates[cpi_release_dates > x].min() - x).days)

# Days to next Non-Farm Payroll data

In [63]:
nfp_release_dates = [
    '2016-01-08', '2016-02-05', '2016-03-04', '2016-04-01', '2016-05-06', '2016-06-03', '2016-07-08', '2016-08-05', '2016-09-02', '2016-10-07', '2016-11-04', '2016-12-02',
    '2017-01-06', '2017-02-03', '2017-03-10', '2017-04-07', '2017-05-05', '2017-06-02', '2017-07-07', '2017-08-04', '2017-09-01', '2017-10-06', '2017-11-03', '2017-12-08',
    '2018-01-05', '2018-02-02', '2018-03-09', '2018-04-06', '2018-05-04', '2018-06-01', '2018-07-06', '2018-08-03', '2018-09-07', '2018-10-05', '2018-11-02', '2018-12-07',
    '2019-01-04', '2019-02-01', '2019-03-08', '2019-04-05', '2019-05-03', '2019-06-07', '2019-07-05', '2019-08-02', '2019-09-06', '2019-10-04', '2019-11-01', '2019-12-06',
    '2020-01-10', '2020-02-07', '2020-03-06', '2020-04-03', '2020-05-08', '2020-06-05', '2020-07-02', '2020-08-07', '2020-09-04', '2020-10-02', '2020-11-06', '2020-12-04',
    '2021-01-08', '2021-02-05', '2021-03-05', '2021-04-02', '2021-05-07', '2021-06-04', '2021-07-02', '2021-08-06', '2021-09-03', '2021-10-08', '2021-11-05', '2021-12-03',
    '2022-01-07', '2022-02-04', '2022-03-04', '2022-04-01', '2022-05-06', '2022-06-03', '2022-07-08', '2022-08-05', '2022-09-02', '2022-10-07', '2022-11-04', '2022-12-02',
    '2023-01-06', '2023-02-03', '2023-03-10', '2023-04-07', '2023-05-05', '2023-06-02', '2023-07-07', '2023-08-04', '2023-09-01', '2023-10-06', '2023-11-03', '2023-12-01',
    '2024-01-05', '2024-02-02', '2024-03-08', '2024-04-05', '2024-05-03', '2024-06-07', '2024-07-05', '2024-08-02', '2024-09-06', '2024-10-04', '2024-11-01', '2024-12-06'
]

nfp_release_dates = pd.to_datetime(nfp_release_dates)

df['Days to Next NFP']\
      = df['Date'].apply(lambda x: (nfp_release_dates[nfp_release_dates > x].min() - x).days)
